# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer


## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [7]:
DATA_DIR = Path("../데이터/open")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)

train_labels: (26304, 4)
sample_submission: (8760, 5)
ldaps_train: (420864, 35) gfs_train: (236736, 40)
ldaps_test: (140160, 35) gfs_test: (78840, 40)


## 3. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [8]:
import warnings
warnings.filterwarnings('ignore')

def aggregate_weather(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    
    if prefix == "ldaps":
        u10 = df['heightAboveGround_10_10u']
        v10 = df['heightAboveGround_10_10v']
        df['ldaps_ws_10'] = np.sqrt(u10**2 + v10**2)
        df['ldaps_ws_cubed_10'] = df['ldaps_ws_10'] ** 3
        df['ldaps_u10_raw'] = u10
        df['ldaps_v10_raw'] = v10
        
        ubl = df['heightAboveGround_5_XBLWS']
        vbl = df['heightAboveGround_5_YBLWS']
        df['ldaps_ws_bl'] = np.sqrt(ubl**2 + vbl**2)
        df['ldaps_ws_cubed_bl'] = df['ldaps_ws_bl'] ** 3
        df['ldaps_ubl_raw'] = ubl
        df['ldaps_vbl_raw'] = vbl

    elif prefix == "gfs":
        u10 = df['heightAboveGround_10_10u']
        v10 = df['heightAboveGround_10_10v']
        df['gfs_ws_10'] = np.sqrt(u10**2 + v10**2)
        df['gfs_ws_cubed_10'] = df['gfs_ws_10'] ** 3
        df['gfs_u10_raw'] = u10
        df['gfs_v10_raw'] = v10
        
        u80 = df['heightAboveGround_80_u']
        v80 = df['heightAboveGround_80_v']
        df['gfs_ws_80'] = np.sqrt(u80**2 + v80**2)
        df['gfs_ws_cubed_80'] = df['gfs_ws_80'] ** 3
        df['gfs_u80_raw'] = u80
        df['gfs_v80_raw'] = v80
        
        u100 = df['heightAboveGround_100_100u']
        v100 = df['heightAboveGround_100_100v']
        df['gfs_ws_100'] = np.sqrt(u100**2 + v100**2)
        df['gfs_ws_cubed_100'] = df['gfs_ws_100'] ** 3
        df['gfs_u100_raw'] = u100
        df['gfs_v100_raw'] = v100

    if prefix == "ldaps":
        df['ldaps_temp_surface'] = df['surface_1_TSRC'] if 'surface_1_TSRC' in df.columns else 0
    elif prefix == "gfs":
        df['gfs_temp_2m'] = df['heightAboveGround_2_2t'] if 'heightAboveGround_2_2t' in df.columns else 0

    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}_mean" for c in agg.columns]
    return agg.reset_index()


def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out


def apply_physics_features(df):
    df = df.copy()
    
    # 1. LDAPS 절대습도 & 공기분압 연산
    T_C_l = df['ldaps_heightAboveGround_2_t_mean']
    RH_l = df['ldaps_heightAboveGround_2_r_mean']
    P_Pa_l = df['ldaps_surface_0_sp_mean']
    P_hPa_l = np.where(P_Pa_l > 10000, P_Pa_l / 100, P_Pa_l)
    T_K_l = T_C_l + 273.15
    
    e_s_l = 6.112 * np.exp((17.67 * T_C_l) / (T_C_l + 243.5))
    e_a_l = e_s_l * (RH_l / 100.0)
    rho_d_l = (P_hPa_l * 100) / (287.058 * T_K_l)
    rho_v_l = (e_a_l * 100) / (461.495 * T_K_l)
    
    df['ldaps_abs_humidity'] = (rho_v_l / (rho_d_l + rho_v_l)) * 1000
    df['ldaps_air_pressure'] = (rho_d_l + rho_v_l) * 287.058 * T_K_l / 100

    # 2. GFS 절대습도 & 공기분압 연산
    T_C_g = df['gfs_heightAboveGround_2_2t_mean']
    RH_g = df['gfs_heightAboveGround_2_2r_mean']
    P_Pa_g = df['gfs_surface_0_sp_mean']
    P_hPa_g = np.where(P_Pa_g > 10000, P_Pa_g / 100, P_Pa_g)
    T_K_g = T_C_g + 273.15
    
    e_s_g = 6.112 * np.exp((17.67 * T_C_g) / (T_C_g + 243.5))
    e_a_g = e_s_g * (RH_g / 100.0)
    rho_d_g = (P_hPa_g * 100) / (287.058 * T_K_g)
    rho_v_g = (e_a_g * 100) / (461.495 * T_K_g)
    
    df['gfs_abs_humidity'] = (rho_v_g / (rho_d_g + rho_v_g)) * 1000
    df['gfs_air_pressure'] = (rho_d_g + rho_v_g) * 287.058 * T_K_g / 100
            
    return df

def add_time_series_features(df):
    df = df.copy()
    
    if 'ldaps_ldaps_temp_surface_mean' in df.columns:
        df['ldaps_density_index'] = 1 / (df['ldaps_ldaps_temp_surface_mean'] + 273.15)
    if 'gfs_gfs_temp_2m_mean' in df.columns:
        df['gfs_density_index'] = 1 / (df['gfs_gfs_temp_2m_mean'] + 273.15)
        
    if 'ldaps_density_index' in df.columns and 'ldaps_ldaps_ws_cubed_10_mean' in df.columns:
        df['ldaps_adj_power'] = df['ldaps_ldaps_ws_cubed_10_mean'] * df['ldaps_density_index']
    if 'gfs_density_index' in df.columns and 'gfs_gfs_ws_cubed_80_mean' in df.columns:
        df['gfs_adj_power'] = df['gfs_gfs_ws_cubed_80_mean'] * df['gfs_density_index']

    target_cols = [
        'ldaps_ldaps_ws_10_mean', 'gfs_gfs_ws_10_mean', 'gfs_gfs_ws_80_mean',
        'ldaps_adj_power', 'gfs_adj_power', 
        'ldaps_abs_humidity', 'gfs_abs_humidity', 'ldaps_air_pressure', 'gfs_air_pressure'
    ]
    
    for col in target_cols:
        if col in df.columns:
            df[f"{col}_lag1"] = df[col].shift(1).bfill()
            df[f"{col}_roll3_mean"] = df[col].rolling(window=3, min_periods=1, center=True).mean()
            df[f"{col}_roll3_std"] = df[col].rolling(window=3, min_periods=1, center=True).std().fillna(0)
        
    return df

# 파이프라인 결합
train_weather = aggregate_weather(ldaps_train, "ldaps").merge(
    aggregate_weather(gfs_train, "gfs"), on="forecast_kst_dtm", how="inner"
)
test_weather = aggregate_weather(ldaps_test, "ldaps").merge(
    aggregate_weather(gfs_test, "gfs"), on="forecast_kst_dtm", how="inner"
)

train_weather = train_weather.sort_values("forecast_kst_dtm").reset_index(drop=True)
test_weather = test_weather.sort_values("forecast_kst_dtm").reset_index(drop=True)

train_weather = apply_physics_features(train_weather)
test_weather = apply_physics_features(test_weather)

train_weather = add_time_series_features(train_weather)
test_weather = add_time_series_features(test_weather)

train_base = train_labels.rename(columns={"kst_dtm": "forecast_kst_dtm"})
train_df = train_base.merge(train_weather, on="forecast_kst_dtm", how="left")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    test_weather, on="forecast_kst_dtm", how="left"
)

X_train = pd.concat(
    [calendar_features(train_df["forecast_kst_dtm"]), train_df.drop(columns=["forecast_kst_dtm", *TARGET_COLS])],
    axis=1,
)
X_test = pd.concat(
    [calendar_features(test_df["forecast_kst_dtm"]), test_df.drop(columns=["forecast_id", "forecast_kst_dtm"])],
    axis=1,
)

print("X_train 최종 변수 주입 완료:", X_train.shape)
print("X_test 최종 변수 주입 완료:", X_test.shape)

X_train 최종 변수 주입 완료: (26304, 131)
X_test 최종 변수 주입 완료: (8760, 131)


In [ ]:
from pycaret.regression import *
import pandas as pd

# 1. 데이터 병합 (PyCaret은 타겟을 포함한 데이터셋을 한 번에 받습니다)
# kpx_group_1로 예시를 듭니다.
target_name = 'kpx_group_1' 
train_data = pd.concat([X_train, train_df[target_name]], axis=1)

# 2. PyCaret 환경 설정
# 주의: fold_strategy='timeseries' 필수!
s = setup(
    data=train_data, 
    target=target_name, 
    fold_strategy='timeseries', 
    fold=5,
    session_id=42,
    use_gpu=True # GPU 사용 가능 시 성능 향상
)

['ldaps_ldaps_u10_raw_mean', 'ldaps_ldaps_v10_raw_mean', 'ldaps_ldaps_ubl_raw_mean', 'ldaps_ldaps_vbl_raw_mean', 'gfs_gfs_u10_raw_mean', 'gfs_gfs_v10_raw_mean', 'gfs_gfs_u80_raw_mean', 'gfs_gfs_v80_raw_mean', 'gfs_gfs_u100_raw_mean', 'gfs_gfs_v100_raw_mean']


## 4. 모델 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [10]:
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# 1. 모델 파라미터 설정
lgb_params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.035,         
    'n_estimators': 3500,          
    'max_depth': 8,                 
    'num_leaves': 63,              
    'colsample_bytree': 0.8,        
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

et_params = {
    'n_estimators': 400, 
    'max_depth': 30, 
    'min_samples_split': 2, 
    'bootstrap': False, 
    'random_state': 42,
    'n_jobs': -1
}

tscv = TimeSeriesSplit(n_splits=5)
predictions = pd.DataFrame(index=sample_submission.index)

# 2. 앙상블 학습 루프 시작
for target in TARGET_COLS:
    train_mask = train_df[target].notna()
    y_train = train_df.loc[train_mask, target].reset_index(drop=True)
    X_train_target = X_train.loc[train_mask].reset_index(drop=True)
    
    print(f"\n========== {target} 앙상블(LGBM+ET) 5-Fold 학습 시작 ==========")
    
    oof_lgbm = np.zeros(len(X_train_target))
    oof_et = np.zeros(len(X_train_target))
    
    test_lgbm = np.zeros(len(X_test))
    test_et = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_target)):
        X_tr, y_tr = X_train_target.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_target.iloc[val_idx], y_train.iloc[val_idx]
        
        # 모델 생성
        model_lgb = lgb.LGBMRegressor(**lgb_params)
        model_et = ExtraTreesRegressor(**et_params)
        
        # 학습
        model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)])
        model_et.fit(X_tr, y_tr)
        
        # 예측
        oof_lgbm[val_idx] = model_lgb.predict(X_va)
        oof_et[val_idx] = model_et.predict(X_va)
        
        test_lgbm += model_lgb.predict(X_test) / tscv.n_splits
        test_et += model_et.predict(X_test) / tscv.n_splits
        
        print(f"Fold {fold+1} 완료")
        
    # 3. 앙상블 및 후처리 (가중치 1.0 적용)
    final_oof = (oof_lgbm + oof_et) / 2
    final_test = (test_lgbm + test_et) / 2
    
    # 예측값 클리핑
    predictions[target] = np.clip(final_test, 0, CAPACITY_KWH[target])
    
    # MAE 확인 (OOF 기준)
    valid_idx = np.where(final_oof != 0)[0]
    mae = mean_absolute_error(y_train.iloc[valid_idx], final_oof[valid_idx])
    print(f"▶▶ [{target}] 앙상블 최종 OOF MAE: {mae:.4f}")


========== kpx_group_1 앙상블(LGBM+ET) 5-Fold 학습 시작 ==========
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
▶▶ [kpx_group_1] 앙상블 최종 OOF MAE: 2241.0121

========== kpx_group_2 앙상블(LGBM+ET) 5-Fold 학습 시작 ==========
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
▶▶ [kpx_group_2] 앙상블 최종 OOF MAE: 2228.9304

========== kpx_group_3 앙상블(LGBM+ET) 5-Fold 학습 시작 ==========
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
▶▶ [kpx_group_3] 앙상블 최종 OOF MAE: 2216.2419


## 5. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [11]:
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for target in TARGET_COLS:
    submission[target] = predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)


     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  16755.621320  18315.549413   
1  forecast_0002  2025-01-01 02:00:00  16612.336918  18416.935242   
2  forecast_0003  2025-01-01 03:00:00  16214.510653  17989.777783   
3  forecast_0004  2025-01-01 04:00:00  15543.224580  16775.923514   
4  forecast_0005  2025-01-01 05:00:00  15323.188927  16847.556463   

    kpx_group_3  
0  17373.561412  
1  16745.604732  
2  15412.089555  
3  13935.627377  
4  12584.886634  
(8760, 5)


## 6. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [12]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")


Saved: ..\데이터\open\baseline_submit.csv


In [13]:
# LDAPS와 GFS의 정확한 기압/습도 컬럼명 확인
print("전체 컬럼 리스트:")
for col in X_train.columns:
    print(col)

전체 컬럼 리스트:
month
day
hour
dayofweek
is_weekend
hour_sin
hour_cos
month_sin
month_cos
ldaps_heightAboveGround_10_10u_mean
ldaps_heightAboveGround_10_10v_mean
ldaps_heightAboveGround_50_50MUmax_mean
ldaps_heightAboveGround_50_50MUmin_mean
ldaps_heightAboveGround_50_50MVmax_mean
ldaps_heightAboveGround_50_50MVmin_mean
ldaps_heightAboveGround_5_XBLWS_mean
ldaps_heightAboveGround_5_YBLWS_mean
ldaps_heightAboveGround_2_t_mean
ldaps_heightAboveGround_2_dpt_mean
ldaps_heightAboveGround_2_r_mean
ldaps_heightAboveGround_2_q_mean
ldaps_surface_0_sp_mean
ldaps_meanSea_0_prmsl_mean
ldaps_etc_0_blh_mean
ldaps_surface_0_NDNSW_mean
ldaps_surface_0_NDNLW_mean
ldaps_heightAboveGround_2_SWDIR_mean
ldaps_heightAboveGround_2_SWDIF_mean
ldaps_etc_0_hcc_mean
ldaps_etc_0_mcc_mean
ldaps_etc_0_lcc_mean
ldaps_etc_0_VLCDC_mean
ldaps_surface_0_avg_lsprate_mean
ldaps_surface_0_lssrate_mean
ldaps_surface_0_ncpcp_mean
ldaps_surface_0_snol_mean
ldaps_surface_0_SNOM_mean
ldaps_surface_0_lsm_mean
ldaps_surface_0_h_mean
